# Kuvantunnistus omilla kuvilla - VGG16 Assisted

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model  # Changed from Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D
import tensorflow as tf
import numpy as np

# Load pre-trained VGG16 model without the top layer
vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the layers of VGG16
for layer in vgg16_base.layers:
    layer.trainable = False

# Create new model on top
x = vgg16_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(6, activation='softmax')(x)

# Create final model
model = Model(inputs=vgg16_base.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Load dataset
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "../datasets/kuvantunnistus",
    image_size=(224, 224),
    batch_size=32,
    label_mode="categorical",
    shuffle=True,
)

# Split dataset
dataset_size = len(dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)

train_dataset = dataset.take(train_size)
validation_dataset = dataset.skip(train_size).take(val_size)
test_dataset = dataset.skip(train_size + val_size)

# Train the model directly (no need for separate feature extraction)
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc}")

Found 478 files belonging to 6 classes.
Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.3753 - loss: 5.6720 - val_accuracy: 0.7969 - val_loss: 0.8034
Epoch 2/10
 4/10 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.6556 - loss: 1.8963 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

# Get the class names
class_names = dataset.class_names

# Evaluate test set accuracy
test_loss, test_acc = model.evaluate(test_dataset)
print(f"\nTest Set Accuracy: {test_acc*100:.2f}%\n")

# Predict only on the test set and collect data
y_pred = []
x_images = []
y_true = []

for images, labels in test_dataset:
    y_pred.extend(model.predict(images, batch_size=32))
    x_images.extend(images.numpy())
    y_true.extend(labels.numpy())

# Convert to numpy arrays
y_pred = np.array(y_pred)
y_true = np.array(y_true)
x_images = np.array(x_images)

# Select 10 random indices
num_samples = 10
random_indices = random.sample(range(len(x_images)), num_samples)

# Plot the selected test images with predictions
for i in random_indices:
    plt.figure(figsize=(10, 4))
    
    # Display image
    plt.subplot(1, 2, 1)
    plt.imshow(x_images[i].astype("uint8"))
    true_class = class_names[np.argmax(y_true[i])]
    plt.title(f"True: {true_class}")
    plt.axis('off')
    
    # Display prediction probabilities
    plt.subplot(1, 2, 2)
    pred_prob = y_pred[i]
    predicted_class = class_names[np.argmax(pred_prob)]
    
    # Highlight correct predictions in green, wrong in red
    colors = ['lightgreen' if (class_names[idx] == true_class) else 'skyblue' 
              for idx in range(len(class_names))]
    
    plt.barh(class_names, pred_prob, color=colors)
    plt.xlabel('Probability')
    plt.title(f"Predicted: {predicted_class}")
    plt.xlim([0, 1])
    
    # Add visual indication if prediction was wrong
    if predicted_class != true_class:
        plt.text(0.5, -0.5, "Incorrect", color='red', 
                ha='center', transform=plt.gca().transAxes)
    else:
        plt.text(0.5, -0.5, "Correct", color='green', 
                ha='center', transform=plt.gca().transAxes)
    
    plt.tight_layout()
    plt.show()